# Spark Exercises 08 — Schemas, Reading & Writing

In pandas you `read_csv` and move on. In Spark — and especially in Foundry — **the schema is a first-class decision**. `inferSchema` is convenient but reads your data an extra time; in production you give Spark an explicit schema. This chapter covers schema-on-read, casting dirty string columns, and writing **Parquet** — including `partitionBy`, the layout that powers fast, partition-pruned reads on real datasets.

**Data files:** `data/orders.csv`, `data/customers.csv`

### 1. **Setup** — open a local `SparkSession` (already written for you). In Foundry the session is created for you; locally we open one ourselves. `F` is the functions module — almost every column expression lives there.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-course")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

### 2. Read `data/orders.csv` with **just** `header=True` (no `inferSchema`). `printSchema()` — **everything is a string**, even `quantity` and `order_ts`. This is the default and it is fast.

### 3. Now read it **with** `inferSchema=True` and `printSchema()`. Convenient — but Spark had to scan the file an **extra time** to guess the types.

### 4. **Best practice: an explicit schema.** Build a `StructType` for the orders file and read with `schema=...` (no inference pass, no surprises). `printSchema()` to confirm.

### 5. Starting from the all-string `raw`, **cast** `quantity` to int and `unit_price` to double, and parse `order_ts` into a real timestamp with `F.to_timestamp(..., "yyyy-MM-dd HH:mm:ss")`. Show the schema and 3 rows.

### 6. Now that `order_ts` is a real timestamp, extract `year` and `month`, and count orders **per month**.

### 7. **Write Parquet.** Create a temp output folder, then write `typed` as Parquet with `mode("overwrite")`. List the folder — note Parquet is a **directory** of part-files, not one file.

### 8. **Read the Parquet back.** Parquet stores the schema inside the files — so no `inferSchema` is needed and types come back exactly. `printSchema()` to prove it.

### 9. **Partitioned write.** Write `typed` as Parquet **partitioned by `channel`** into a new temp path. List the directory — you'll see `channel=web`, `channel=app`, … sub-folders.

### 10. **Partition pruning.** Read the partitioned dataset and filter `channel == 'web'`, then `.explain()`. Look for `PartitionFilters` — Spark reads **only the `channel=web` folder**, skipping the rest.

### 11. Finally, write a single CSV (coalesce to 1 file) of the per-channel order counts, with a header, then read it back to confirm.